In [ ]:
# creating a united healty and tcga atac counts matrix filtered for disease, (then the R script needs to be applied for log CPM an QN)

import pandas as pd
import numpy as np
import os
os.chdir("C:/Users/stavz/Desktop/masters/APM")

TCGA_RAW_ATAC_COUNTS_PATH = "data/TCGA_ATAC/TCGA-ATAC_PanCan_Raw_Counts.txt"
HEALTHY_COUNTS_DIR = "data/TCGA_ATAC/normal_counts"
TCGA_SAMPLES_TABLE_PATH = "data/TCGA.tsv"
ATAC_META_PATH = "data/TCGA_ATAC/TCGA_identifier_mapping.txt"

def filter_TCGA_samples(counts: pd.DataFrame, tcga_samples_table: pd.DataFrame, atac_meta: pd.DataFrame, first_sample_col: int, primary_disease: str) -> pd.DataFrame:
    tcga_samples = tcga_samples_table[tcga_samples_table["_primary_disease"] == primary_disease]["sample"].tolist()
    tcga_samples = ["-".join(s.split("-")[:3]) for s in tcga_samples]
    print("found ", len(tcga_samples), "TCGA samples, matching primary disease: ", primary_disease)
    atac_meta["Case_ID_short"] = atac_meta["Case_ID"].apply(lambda id : "-".join(id.split("-")[:3]))
    atac_samples_bam_prefices = atac_meta[atac_meta["Case_ID_short"].isin(tcga_samples)]["bam_prefix"].tolist()
    atac_samples_bam_prefices = [s.replace("-", "_") for s in atac_samples_bam_prefices]
    samples_copy = counts[atac_samples_bam_prefices].copy()
    return pd.concat([counts.iloc[:,:first_sample_col], samples_copy], axis=1)
        


def add_healthy_counts(counts: pd.DataFrame, vector_folder: str) -> pd.DataFrame:
    for fname in os.listdir(vector_folder):
        if fname.endswith(".vector"):
            sample_name = fname.replace(".vector", "")  # remove extension
            fpath = os.path.join(vector_folder, fname)
            
            # Load the vector (no header)
            vec = pd.read_csv(fpath, header=None).iloc[:, 0]
            
            # Sanity check: same number of rows as TCGA peak set
            if len(vec) != counts.shape[0]:
                raise ValueError(f"Row mismatch in {fname}: expected {counts.shape[0]}, got {len(vec)}")
            
            # Add as new column
            counts[sample_name] = vec.values
            print(f"Added {sample_name}")
    return counts



def harmonize_atac_counts(tcga_raw_atac_counts_path: str, healthy_counts_dir_path: str, tcga_samples_table_path: str, atac_meta_path: str, first_sample_col: int, primary_disease: str, TCGA_CANCER_TYPE: str) -> pd.DataFrame:
    tcga_raw_atac_counts = pd.read_csv(tcga_raw_atac_counts_path, sep="\t")
    tcga_samples_table = pd.read_csv(tcga_samples_table_path, sep="\t")
    atac_meta = pd.read_csv(atac_meta_path, sep="\t")
   

    tcga_filtered_counts = filter_TCGA_samples(tcga_raw_atac_counts, tcga_samples_table, atac_meta, first_sample_col, primary_disease)
    all_samples_counts = add_healthy_counts(tcga_filtered_counts, healthy_counts_dir_path)
    all_samples_counts.to_csv(f"data/TCGA_ATAC/TCGA_BRCA_ATAC_with_normals_raw.txt", sep="\t", index=False)



test= harmonize_atac_counts(TCGA_RAW_ATAC_COUNTS_PATH, HEALTHY_COUNTS_DIR, TCGA_SAMPLES_TABLE_PATH, ATAC_META_PATH, 7, "breast invasive carcinoma", "BRCA")





found  1241 TCGA samples, matching primary disease:  breast invasive carcinoma
Added ENCFF112BEA
Added ENCFF344HVC
Added ENCFF504NFQ


In [ ]:
# collapsing replicates to case level

import pandas as pd

def collapse_atac_replicates(
    log_cpm_qn: pd.DataFrame,
    atac_meta_path: str,
    first_sample_col: int,
    bam_prefix_col: str = "bam_prefix",
    case_id_col: str = "Case_ID",
) -> pd.DataFrame:
    """
    Collapse technical ATAC replicates to case-level by averaging
    the quantile-normalized log2-CPM values.

    Parameters
    ----------
    log_cpm_qn : pd.DataFrame
        DataFrame with meta columns in [0:first_sample_col)
        and quantile-normalized log2-CPM in [first_sample_col:].
        Columns in the expression part are sample/library names
        (e.g. bam_prefix with '-' replaced by '_').
    atac_meta_path : str
        Path to TCGA ATAC meta table (e.g. TCGA_identifier_mapping.txt)
        containing bam_prefix and Case_ID columns.
    first_sample_col : int
        Index of the first sample column (meta columns are before this).
    bam_prefix_col : str
        Column name in atac_meta containing bam prefix (with '-').
    case_id_col : str
        Column name in atac_meta containing case ID.

    Returns
    -------
    pd.DataFrame
        DataFrame with the same meta columns, and columns collapsed
        so that each TCGA Case_ID appears once (replicates averaged),
        while non-TCGA samples (e.g. ENCODE normals) remain unchanged.
    """

    # Split meta vs expression
    meta_qn = log_cpm_qn.iloc[:, :first_sample_col]
    expr_qn = log_cpm_qn.iloc[:, first_sample_col:]

    # Load ATAC meta
    atac_meta = pd.read_csv(atac_meta_path, sep="\t")

    # Make a normalized bam_prefix that matches column names
    # (your columns use '_' where bam_prefix has '-')
    atac_meta["bam_prefix_norm"] = atac_meta[bam_prefix_col].str.replace("-", "_")

    # Keep only libraries that actually exist in the expression matrix
    atac_meta = atac_meta[atac_meta["bam_prefix_norm"].isin(expr_qn.columns)]

    # Build mapping: column name (with '_') -> Case_ID
    col_to_case = dict(zip(atac_meta["bam_prefix_norm"], atac_meta[case_id_col]))

    # For every column in expr_qn (including ENCODE normals),
    # use Case_ID if available, otherwise keep its own name
    group_keys = [col_to_case.get(c, c) for c in expr_qn.columns]

    # Collapse replicates: transpose -> groupby columns -> mean -> transpose back
    collapsed_expr = expr_qn.T.groupby(group_keys).mean().T

    # Reattach meta columns
    full_collapsed = pd.concat([meta_qn, collapsed_expr], axis=1)

    return full_collapsed
log_cpm_qn = pd.read_csv(
    "data/TCGA_ATAC/TCGA_BRCA_ATAC_with_normals_logCPM_QN.txt",
    sep="\t"
)

collapsed = collapse_atac_replicates(
    log_cpm_qn=log_cpm_qn,
    atac_meta_path=ATAC_META_PATH,
    first_sample_col=7,
)

collapsed.to_csv(
    "data/TCGA_ATAC/TCGA_NORMAL_LOG_CPM_QN_BRCA_case_level.csv",
    index=False
)


In [2]:
import pandas as pd
normals_manifest = pd.DataFrame({"sample_id": ["ENCFF112BEA", "ENCFF344HVC", "ENCFF504NFQ"], 
"bam_path": ["data/TCGA_ATAC/normal_bams/ENCFF112BEA.bam", "data/TCGA_ATAC/normal_bams/ENCFF344HVC.bam", "data/TCGA_ATAC/normal_bams/ENCFF504NFQ.bam"]})
normals_manifest.to_csv("scripts/scripts/ATAC_scripts/normals_manifest.csv", index=False)